# Stacking Tabular Foundation Models (TFMs) with 124 Public OOF/Test prediction pairs

This notebook stacks **9 tabular foundation models** plus tuned XGBoost and RealMLP baselines (from Kaggle dataset [here][2]) with **124 public** OOF/test prediction pairs (from version 3 public notebook [here][1])

- Every pair is an out-of-fold prediction for `train.csv` plus a test prediction, from `StratifiedKFold(5, shuffle=True, random_state=42)`.
- Probabilities are converted to logits. A few pairs are **already logits** (values outside 0–1); they are detected and used as they are.
- The stacker is an L2 logistic regression on GPU (**cuML**), trained per fold on the same 5 folds.
- Outputs: stack CV AUC, the top models by coefficient, `stack_oof.npy`, `stack_test.npy` and `submission.csv`.

Enable a **GPU** accelerator for cuML.

[1]: https://www.kaggle.com/code/kratosyan/s6e10-stacking-every-public-oof-library?scriptVersionId=355670407
[2]: https://www.kaggle.com/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions

# Sources (please upvote them)

| source | what it contributes |
|---|---|
| [goodpjw2008 · Part 8](https://www.kaggle.com/code/goodpjw2008/s6e10-tabpfn-route-categories-lb-0-96160) | 48 members: GBDT / RealMLP / DeepFM on route-ID features, plus 8 TabPFN-3.5 and TabICL members |
| [sachith7 · stack OOF predictions](https://www.kaggle.com/datasets/sachith7/s6e10-stack-oof-predictions) | 22 members, 10 of them trained on 10 folds |
| [megayak · OOF library](https://www.kaggle.com/datasets/megayak/s6e10-oof-library) | LightGBM / XGBoost / CatBoost / RealMLP (seed groups averaged → 11) |
| [dariushafshar · golem OOF library](https://www.kaggle.com/datasets/dariushafshar/s6e10-golem-oof-library) | 14 members |
| [arhancanli12 · OOF + test predictions](https://www.kaggle.com/datasets/arhancanli12/s6e10-oof-test-predictions) | LGB / XGB / CatBoost / RealMLP |
| [najiama · OOF](https://www.kaggle.com/datasets/najiama/s6e10-oof) | blend 01 (blend 02 adds nothing on top) |
| [samanyu1808 · TabPFN route + og](https://www.kaggle.com/code/samanyu1808/s6e10-tabpfn-route-og), [TabPFN seeds](https://www.kaggle.com/code/samanyu1808/s6e10-tabpfn-seeds) | TabPFN-3.5 multi-seed |
| [hemingweb · TabPFN-3.5 full context](https://www.kaggle.com/code/hemingweb/s6e10-exp01-tabpfn35-full-context) | TabPFN-3.5, raw columns |
| [mitudru · equality blocks](https://www.kaggle.com/code/mitudru/s6e10-analysis-first-equality-blocks-stack) | 7 members |
| [wangxintong111 · CatBoost rating crosses](https://www.kaggle.com/code/wangxintong111/s6e10-catboost-rating-crosses-cv-0-96080), [LightGBM context stats](https://www.kaggle.com/code/wangxintong111/s6e10-lightgbm-context-stats-cv-0-96065) | 2 members |
| [thisray · XGB / RealMLP / CatBoost](https://www.kaggle.com/code/thisray/s6e10-xgb-realmlp-catboost-lb-0-96123) | 3 members |
| [megayak · honest OOF stack](https://www.kaggle.com/code/megayak/s6e10-honest-oof-stack) | 4 members |
| [busyaprime · Route or distance? Both](https://www.kaggle.com/code/busyaprime/s6e10-route-or-distance-both-lb-0-96169) | 5 members (2 LightGBM, CatBoost, XGBoost, RealMLP), OOF saved by re-running their notebook: [dataset](https://www.kaggle.com/datasets/kratosyan/s6e10-busyaprime-five-members-oof) |
| [cdeotte · TFMs](https://www.kaggle.com/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions) | 11 members (9 TFMs, baseline XGBoost, baseline RealMLP)|

Notebook outputs are pinned to the version attached to this notebook; if an author re-runs theirs, the numbers can move
in the 5th decimal. The TabPFN-3.5 members come with Prior Labs' non-commercial licence; their authors note that it
allows use in Kaggle competitions.

[1]: https://www.kaggle.com/code/kratosyan/s6e10-stacking-every-public-oof-library?scriptVersionId=355670407

## Configuration

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from cuml.linear_model import LogisticRegression

COMP = "/kaggle/input/competitions/playground-series-s6e10"
PRED_DIRS = [
    "/kaggle/input/datasets/cdeotte/s6e10-124-public-oof-and-test-predictions",
    "/kaggle/input/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions",
]
OUT = "/kaggle/working"
TARGET = "satisfaction"
TOP_N = 20  # number of models to list by |logistic regression coefficient|
C = 1.0     # inverse L2 regularization strength of the stacker

## Labels and folds

The same `StratifiedKFold(5, shuffle=True, random_state=42)` folds that produced the OOF predictions.

In [2]:
train = pd.read_csv(f"{COMP}/train.csv", usecols=["id", TARGET])
test = pd.read_csv(f"{COMP}/test.csv", usecols=["id"])
y = train[TARGET].astype(int).to_numpy()

folds = np.zeros(len(y), dtype=int)
for k, (_, valid_idx) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y)):
    folds[valid_idx] = k
print(f"Train {len(train):,} rows | Test {len(test):,} rows | positive rate {y.mean():.4f}")

Train 699,635 rows | Test 299,844 rows | positive rate 0.4436


## Load every OOF/test pair

Each `oof_<model>.npy` is matched with `test_<model>.npy`. Probabilities become logits (clipped at 1e-15, then at ±30).

A pair is treated as **already logits** if any OOF or test value lies outside 0–1. Those pairs are not converted.

In [3]:
def to_logit(p):
    p = np.clip(p.astype(np.float64), 1e-15, 1 - 1e-15)
    return np.log(p / (1 - p))

names, X_oof, X_test, rows = [], [], [], []
for d in PRED_DIRS:
    for f in sorted(glob.glob(f"{d}/oof_*.npy")):
        name = os.path.basename(f)[len("oof_"):-len(".npy")]
        oof, tst = np.load(f), np.load(f"{d}/test_{name}.npy")
        assert oof.shape == (len(train),) and tst.shape == (len(test),), name
        assert np.isfinite(oof).all() and np.isfinite(tst).all(), name
        already_logit = bool(oof.min() < 0 or oof.max() > 1 or tst.min() < 0 or tst.max() > 1)
        if not already_logit:
            oof, tst = to_logit(oof), to_logit(tst)
        names.append(name)
        X_oof.append(np.clip(oof, -30, 30).astype(np.float32))
        X_test.append(np.clip(tst, -30, 30).astype(np.float32))
        rows.append(dict(model=name, dataset=os.path.basename(d), already_logit=already_logit,
                         oof_auc=roc_auc_score(y, X_oof[-1])))

assert len(set(names)) == len(names), "duplicate model names"
X_oof, X_test = np.stack(X_oof, axis=1), np.stack(X_test, axis=1)
models = pd.DataFrame(rows)
print(f"Loaded {len(names)} OOF/test pairs: X_oof {X_oof.shape}, X_test {X_test.shape}")
print(models["dataset"].value_counts().to_string())
print("\nAlready logits (used as is, not converted):")
print(models[models["already_logit"]].to_string(index=False))
print("\nBest single models by OOF AUC:")
models.sort_values("oof_auc", ascending=False).head(10).reset_index(drop=True)

Loaded 135 OOF/test pairs: X_oof (699635, 135), X_test (299844, 135)
dataset
s6e10-124-public-oof-and-test-predictions    124
s6e10-tfm-oof-and-test-predictions            11

Already logits (used as is, not converted):
        model                                   dataset  already_logit  oof_auc
    ar_cat_5f s6e10-124-public-oof-and-test-predictions           True 0.960940
    ar_lgb_5f s6e10-124-public-oof-and-test-predictions           True 0.961029
ar_realmlp_5f s6e10-124-public-oof-and-test-predictions           True 0.960879
    ar_xgb_5f s6e10-124-public-oof-and-test-predictions           True 0.961049

Best single models by OOF AUC:


,model,dataset,already_logit,oof_auc
0,nj_01,s6e10-124-public-oof-and-test-predictions,False,0.961744
1,sa_rmlp_v4aux_k10s2,s6e10-124-public-oof-and-test-predictions,False,0.961413
2,sa_rmlp_v4aux_k10,s6e10-124-public-oof-and-test-predictions,False,0.961398
3,sa_rmlp_B_k10,s6e10-124-public-oof-and-test-predictions,False,0.961378
4,gp_realmlp_vb_aux,s6e10-124-public-oof-and-test-predictions,False,0.961345
5,gp_realmlp_vb_aux_s2,s6e10-124-public-oof-and-test-predictions,False,0.961316
6,mi_realmlp_lean,s6e10-124-public-oof-and-test-predictions,False,0.961307
7,mi_xgb_d5,s6e10-124-public-oof-and-test-predictions,False,0.961288
8,sa_rmlp_B,s6e10-124-public-oof-and-test-predictions,False,0.961282
9,pfn_samanyu_route_og,s6e10-124-public-oof-and-test-predictions,False,0.961268


## Stack with cuML logistic regression

For each fold, an L2 logistic regression is trained on the other folds' OOF logits and predicts the held-out fold. Test predictions are the mean of the 5 fold models.

Note: cuML's solver is not bit-for-bit deterministic, so CV can differ by about 1e-6 between runs. cuML may print `L-BFGS line search failed` warnings; they are expected here and harmless (the solver stops at its last valid step).

In [4]:
oof_stack = np.zeros(len(y))
test_stack = np.zeros(len(test))
coefs = []
for k in range(5):
    tr, va = folds != k, folds == k
    lr = LogisticRegression(penalty="l2", C=C, fit_intercept=True, solver="qn", max_iter=2000, tol=1e-7,
                            output_type="numpy", verbose=False)
    lr.fit(X_oof[tr], y[tr])
    oof_stack[va] = lr.predict_proba(X_oof[va])[:, 1]
    test_stack += lr.predict_proba(X_test)[:, 1] / 5
    coefs.append(np.ravel(lr.coef_))
    print(f"Fold {k} AUC: {roc_auc_score(y[va], oof_stack[va]):.6f}")

print(f"\nStack CV AUC: {roc_auc_score(y, oof_stack):.6f} ({len(names)} models)")

[2026-10-07 04:16:50.029] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 0 AUC: 0.962180
[2026-10-07 04:16:54.629] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 1 AUC: 0.961215
[2026-10-07 04:16:56.292] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 2 AUC: 0.962661
[2026-10-07 04:16:58.064] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 3 AUC: 0.962332
[2026-10-07 04:16:59.320] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 4 AUC: 0.962019

Stack CV AUC: 0.962076 (135 models)


## Most important models

The `TOP_N` models with the largest absolute coefficient, averaged over the 5 fold models.

Coefficients are per unit of logit. Members on a different scale, such as the already-logit pairs, get proportionally different coefficients.

In [5]:
coef = np.mean(coefs, axis=0)
importance = models.assign(coefficient=coef, abs_coefficient=np.abs(coef))
importance = importance.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
print(f"Top {TOP_N} of {len(names)} models by |coefficient|:")
importance.head(TOP_N)

Top 20 of 135 models by |coefficient|:


,model,dataset,already_logit,oof_auc,coefficient,abs_coefficient
0,tabfm,s6e10-tfm-oof-and-test-predictions,False,0.958871,0.288873,0.288873
1,bp_realmlp,s6e10-124-public-oof-and-test-predictions,False,0.961216,0.266111,0.266111
2,limix2,s6e10-tfm-oof-and-test-predictions,False,0.958474,0.229347,0.229347
3,causilo,s6e10-tfm-oof-and-test-predictions,False,0.958275,-0.198110,0.198110
4,sa_rmlp_ogx_aux,s6e10-124-public-oof-and-test-predictions,False,0.961081,-0.130650,0.130650
5,sa_rmlp_raw_aux_k10,s6e10-124-public-oof-and-test-predictions,False,0.961252,0.120083,0.120083
6,gp_realmlp_pub_aux,s6e10-124-public-oof-and-test-predictions,False,0.960914,-0.118810,0.118810
7,gp_realmlp_pub_s1,s6e10-124-public-oof-and-test-predictions,False,0.960718,-0.104878,0.104878
8,sa_xgb_A_auxev,s6e10-124-public-oof-and-test-predictions,False,0.961044,-0.101467,0.101467
9,gp_tabpfn_catfd10,s6e10-124-public-oof-and-test-predictions,False,0.961071,0.100119,0.100119


## Save stack predictions and submission

In [6]:
np.save(f"{OUT}/stack_oof.npy", oof_stack)
np.save(f"{OUT}/stack_test.npy", test_stack)

submission = pd.read_csv(f"{COMP}/sample_submission.csv")
assert np.array_equal(submission["id"].to_numpy(), test["id"].to_numpy())
submission[TARGET] = test_stack
submission.to_csv(f"{OUT}/submission.csv", index=False)
submission.head()

,id,satisfaction
0,699635,0.263437
1,699636,0.706651
2,699637,0.920529
3,699638,0.087549
4,699639,0.045340


In [7]:
print("\nBest single models by OOF AUC:")
print(
    models.sort_values("oof_auc", ascending=False)
          .head(20)
          .to_string(index=False)
)


Best single models by OOF AUC:
               model                                   dataset  already_logit  oof_auc
               nj_01 s6e10-124-public-oof-and-test-predictions          False 0.961744
 sa_rmlp_v4aux_k10s2 s6e10-124-public-oof-and-test-predictions          False 0.961413
   sa_rmlp_v4aux_k10 s6e10-124-public-oof-and-test-predictions          False 0.961398
       sa_rmlp_B_k10 s6e10-124-public-oof-and-test-predictions          False 0.961378
   gp_realmlp_vb_aux s6e10-124-public-oof-and-test-predictions          False 0.961345
gp_realmlp_vb_aux_s2 s6e10-124-public-oof-and-test-predictions          False 0.961316
     mi_realmlp_lean s6e10-124-public-oof-and-test-predictions          False 0.961307
           mi_xgb_d5 s6e10-124-public-oof-and-test-predictions          False 0.961288
           sa_rmlp_B s6e10-124-public-oof-and-test-predictions          False 0.961282
pfn_samanyu_route_og s6e10-124-public-oof-and-test-predictions          False 0.961268
       sa_r

In [8]:
import shutil

shutil.copy(
    "/kaggle/working/submission.csv",
    "/kaggle/working/submission_135model_stack_C1.csv"
)

print("Saved:")
print("/kaggle/working/submission_135model_stack_C1.csv")

Saved:
/kaggle/working/submission_135model_stack_C1.csv


In [10]:
import pandas as pd
import os
import glob

print("Submission files:")
for f in glob.glob("/kaggle/working/**/*.csv", recursive=True):
    if "submission" in os.path.basename(f).lower():
        df = pd.read_csv(f)
        print(
            os.path.basename(f),
            "shape=", df.shape,
            "mean=", df[TARGET].mean(),
            "min=", df[TARGET].min(),
            "max=", df[TARGET].max()
        )

Submission files:
submission.csv shape= (299844, 2) mean= 0.44349885363821384 min= 0.0046022539027035 max= 0.9982352018356324
submission_135model_stack_C1.csv shape= (299844, 2) mean= 0.44349885363821384 min= 0.0046022539027035 max= 0.9982352018356324


In [11]:
files = [
    f for f in glob.glob("/kaggle/working/**/*.csv", recursive=True)
    if "submission" in os.path.basename(f).lower()
]

for i in range(len(files)):
    for j in range(i + 1, len(files)):
        a = pd.read_csv(files[i])
        b = pd.read_csv(files[j])

        p1 = a[TARGET].to_numpy()
        p2 = b[TARGET].to_numpy()

        print("\n", os.path.basename(files[i]), "vs", os.path.basename(files[j]))
        print("Same predictions:", np.array_equal(p1, p2))
        print("Correlation:", np.corrcoef(p1, p2)[0, 1])
        print("Max absolute diff:", np.max(np.abs(p1 - p2)))


 submission.csv vs submission_135model_stack_C1.csv
Same predictions: True
Correlation: 0.9999999999999999
Max absolute diff: 0.0


In [14]:
stack_test = np.load("/kaggle/working/stack_test.npy")
stack_oof = np.load("/kaggle/working/stack_oof.npy")

print("\nStack test:")
print("shape:", stack_test.shape)
print("mean:", stack_test.mean())
print("min:", stack_test.min())
print("max:", stack_test.max())

for f in files:
    sub = pd.read_csv(f)
    p = sub[TARGET].to_numpy()

    print("\n", os.path.basename(f))
    print("Correlation with stack_test:",
          np.corrcoef(p, stack_test)[0, 1])
    print("Max abs diff:",
          np.max(np.abs(p - stack_test)))
    print("Exactly same:",
          np.array_equal(p, stack_test))


Stack test:
shape: (299844,)
mean: 0.44349885363821384
min: 0.004602253902703524
max: 0.9982352018356323

 submission.csv
Correlation with stack_test: 1.0
Max abs diff: 1.1102230246251565e-16
Exactly same: False

 submission_135model_stack_C1.csv
Correlation with stack_test: 1.0
Max abs diff: 1.1102230246251565e-16
Exactly same: False


In [15]:
# ============================================================
# LOCK CURRENT CHAMPION
# ============================================================

CHAMPION_OOF = stack_oof.copy()
CHAMPION_TEST = stack_test.copy()

print("Champion locked")
print("OOF shape :", CHAMPION_OOF.shape)
print("TEST shape:", CHAMPION_TEST.shape)
print("OOF AUC   :", roc_auc_score(y, CHAMPION_OOF))
print("TEST mean :", CHAMPION_TEST.mean())

Champion locked
OOF shape : (699635,)
TEST shape: (299844,)
OOF AUC   : 0.9620761595140688
TEST mean : 0.44349885363821384


In [16]:
# ============================================================
# PHASE 8B-1 — CHAMPION RESIDUAL / DIVERSITY ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

anchor = CHAMPION_OOF

results = []

for i, name in enumerate(names):

    pred = X_oof[:, i]

    # Correlation with champion
    corr = np.corrcoef(anchor, pred)[0, 1]

    # Residual relative to champion
    residual = pred - anchor

    residual_std = residual.std()

    # How much the model differs from champion
    mean_abs_diff = np.mean(np.abs(residual))

    # Individual AUC
    individual_auc = roc_auc_score(y, pred)

    results.append({
        "model": name,
        "individual_auc": individual_auc,
        "champion_corr": corr,
        "mean_abs_diff": mean_abs_diff,
        "residual_std": residual_std
    })

diversity = pd.DataFrame(results)

print("=" * 80)
print("PHASE 8B-1 — CHAMPION DIVERSITY")
print("=" * 80)

print("\nMost diverse models:")
print(
    diversity
    .sort_values("champion_corr")
    .head(25)
    .to_string(index=False)
)

print("\nStrong + diverse models:")

strong_diverse = diversity[
    (diversity["individual_auc"] >= 0.9600)
].sort_values(
    ["champion_corr", "individual_auc"],
    ascending=[True, False]
)

print(
    strong_diverse
    .head(30)
    .to_string(index=False)
)

PHASE 8B-1 — CHAMPION DIVERSITY

Most diverse models:
              model  individual_auc  champion_corr  mean_abs_diff  residual_std
              nj_01        0.961744       0.813624       1.090721      1.482577
               gl_f        0.948865       0.917253       2.452846      2.646383
               gl_g        0.954592       0.964224       2.053860      2.089938
       sa_xgb_route        0.960317       0.979756       2.581989      2.642694
               gl_e        0.956817       0.981456       2.499442      2.523636
            meh_xgb        0.960432       0.981967       2.570348      2.619836
       gp_deepfm_v1        0.960728       0.982069       2.646482      2.707294
   gp_deepfm_rawaux        0.960479       0.982160       2.648740      2.704584
          sa_xgb_te        0.960179       0.982218       2.563111      2.610536
               gl_b        0.958727       0.982618       2.552739      2.586887
            meh_cat        0.960329       0.982743       2.568089 

In [17]:
# ============================================================
# PHASE 8B-2 — NJ_01 vs CHAMPION
# ============================================================

from sklearn.metrics import roc_auc_score
import numpy as np

idx = names.index("nj_01")

nj_oof = X_oof[:, idx]
nj_test = X_test[:, idx]

print("=" * 70)
print("PHASE 8B-2 — NJ_01 vs CHAMPION")
print("=" * 70)

print(f"Champion AUC : {roc_auc_score(y, CHAMPION_OOF):.9f}")
print(f"NJ_01 AUC    : {roc_auc_score(y, nj_oof):.9f}")
print(f"Correlation  : {np.corrcoef(CHAMPION_OOF, nj_oof)[0,1]:.9f}")

print("\nBlend results:")
print("-" * 50)

blend_results = []

for w in np.arange(0.00, 1.01, 0.025):

    pred = (1-w) * CHAMPION_OOF + w * nj_oof
    auc = roc_auc_score(y, pred)

    blend_results.append((w, auc))

    print(
        f"NJ weight={w:5.3f} | "
        f"Champion weight={1-w:5.3f} | "
        f"AUC={auc:.9f} | "
        f"Delta={auc-roc_auc_score(y, CHAMPION_OOF):+.9f}"
    )

best_w, best_auc = max(blend_results, key=lambda x: x[1])

print("\n" + "=" * 70)
print(f"BEST NJ_01 WEIGHT : {best_w:.3f}")
print(f"BEST AUC          : {best_auc:.9f}")
print(f"DELTA             : {best_auc - roc_auc_score(y, CHAMPION_OOF):+.9f}")
print("=" * 70)

PHASE 8B-2 — NJ_01 vs CHAMPION
Champion AUC : 0.962076160
NJ_01 AUC    : 0.961744050
Correlation  : 0.813624326

Blend results:
--------------------------------------------------
NJ weight=0.000 | Champion weight=1.000 | AUC=0.962076160 | Delta=+0.000000000
NJ weight=0.025 | Champion weight=0.975 | AUC=0.962044244 | Delta=-0.000031915
NJ weight=0.050 | Champion weight=0.950 | AUC=0.962005056 | Delta=-0.000071104
NJ weight=0.075 | Champion weight=0.925 | AUC=0.961979520 | Delta=-0.000096640
NJ weight=0.100 | Champion weight=0.900 | AUC=0.961961026 | Delta=-0.000115134
NJ weight=0.125 | Champion weight=0.875 | AUC=0.961946971 | Delta=-0.000129189
NJ weight=0.150 | Champion weight=0.850 | AUC=0.961935528 | Delta=-0.000140632
NJ weight=0.175 | Champion weight=0.825 | AUC=0.961925966 | Delta=-0.000150194
NJ weight=0.200 | Champion weight=0.800 | AUC=0.961917693 | Delta=-0.000158467
NJ weight=0.225 | Champion weight=0.775 | AUC=0.961910369 | Delta=-0.000165791
NJ weight=0.250 | Champion weig

In [18]:
# ============================================================
# PHASE 8C — STACK COEFFICIENT + AUC ANALYSIS
# ============================================================

coef_df = models.copy()

coef_df["coefficient"] = coef
coef_df["abs_coefficient"] = np.abs(coef)

coef_df["auc_rank"] = (
    coef_df["oof_auc"]
    .rank(ascending=False, method="min")
)

coef_df["coef_rank"] = (
    coef_df["abs_coefficient"]
    .rank(ascending=False, method="min")
)

print("=" * 80)
print("PHASE 8C — STACK COEFFICIENT ANALYSIS")
print("=" * 80)

print("\nStrongest positive coefficients:")
print(
    coef_df
    .sort_values("coefficient", ascending=False)
    .head(20)
    [["model", "oof_auc", "coefficient"]]
    .to_string(index=False)
)

print("\nStrongest negative coefficients:")
print(
    coef_df
    .sort_values("coefficient")
    .head(20)
    [["model", "oof_auc", "coefficient"]]
    .to_string(index=False)
)

print("\nSmallest absolute coefficients:")
print(
    coef_df
    .sort_values("abs_coefficient")
    .head(30)
    [["model", "oof_auc", "coefficient"]]
    .to_string(index=False)
)

PHASE 8C — STACK COEFFICIENT ANALYSIS

Strongest positive coefficients:
                model  oof_auc  coefficient
                tabfm 0.958871     0.288873
           bp_realmlp 0.961216     0.266111
               limix2 0.958474     0.229347
  sa_rmlp_raw_aux_k10 0.961252     0.120083
    gp_tabpfn_catfd10 0.961071     0.100119
sa_cat_ctr_auxall_k10 0.961174     0.095874
    pfn_hemingweb_raw 0.961053     0.083520
 pfn_samanyu_route_og 0.961268     0.081524
         sa_xgb_route 0.960317     0.078594
  sa_rmlp_v4aux_k10s2 0.961413     0.074614
        sa_rmlp_B_k10 0.961378     0.071564
           mg_cat_ctr 0.960980     0.067494
  sa_rmlp_v4ev_e4_k10 0.961244     0.066548
      gp_tabpfn_catfd 0.960925     0.062041
             tabpfn35 0.961145     0.058846
            mi_xgb_d5 0.961288     0.058278
            kumolarge 0.960036     0.057826
              bp_lgbm 0.961058     0.052947
                 gl_m 0.960411     0.052143
           gp_cat_ctr 0.961045     0.051997

Str

In [19]:
# ============================================================
# PHASE 8D — COEFFICIENT-BASED STACK PRUNING
# ============================================================

from cuml.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Rank models by absolute coefficient
ranked_idx = np.argsort(np.abs(coef))[::-1]

sizes = [100, 75, 50, 30, 20, 15, 10]

pruning_results = []

print("=" * 80)
print("PHASE 8D — COEFFICIENT-BASED STACK PRUNING")
print("=" * 80)

for n_models in sizes:

    selected = ranked_idx[:n_models]

    X_sel = X_oof[:, selected]

    oof_pred = np.zeros(len(y))

    for k in range(5):

        tr = folds != k
        va = folds == k

        lr = LogisticRegression(
            penalty="l2",
            C=C,
            fit_intercept=True,
            solver="qn",
            max_iter=2000,
            tol=1e-7,
            output_type="numpy",
            verbose=False
        )

        lr.fit(X_sel[tr], y[tr])

        oof_pred[va] = lr.predict_proba(X_sel[va])[:, 1]

    auc = roc_auc_score(y, oof_pred)

    pruning_results.append({
        "models": n_models,
        "auc": auc,
        "delta_vs_135": auc - roc_auc_score(y, CHAMPION_OOF)
    })

    print(
        f"{n_models:3d} models | "
        f"AUC = {auc:.9f} | "
        f"Delta = {auc - roc_auc_score(y, CHAMPION_OOF):+.9f}"
    )

print("\n" + "=" * 80)
print("RESULTS")
print("=" * 80)

print(pd.DataFrame(pruning_results).to_string(index=False))

PHASE 8D — COEFFICIENT-BASED STACK PRUNING
[2026-10-07 04:49:17.960] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:49:19.817] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:49:22.586] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:49:24.115] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
100 models | AUC = 0.962091172 | Delta = +0.000015012
[2026-10-07 04:49:26.385] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:49:28.193] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:49:29.591] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:49:30.216] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:

In [20]:
# ============================================================
# PHASE 8E — FINE PRUNING AROUND THE 50-MODEL SWEET SPOT
# ============================================================

sizes = [65, 60, 55, 50, 45, 40, 35, 30]

fine_results = []

for n_models in sizes:

    selected = ranked_idx[:n_models]
    X_sel = X_oof[:, selected]

    oof_pred = np.zeros(len(y))

    for k in range(5):

        tr = folds != k
        va = folds == k

        lr = LogisticRegression(
            penalty="l2",
            C=C,
            fit_intercept=True,
            solver="qn",
            max_iter=2000,
            tol=1e-7,
            output_type="numpy",
            verbose=False
        )

        lr.fit(X_sel[tr], y[tr])
        oof_pred[va] = lr.predict_proba(X_sel[va])[:, 1]

    auc = roc_auc_score(y, oof_pred)

    fine_results.append({
        "models": n_models,
        "auc": auc,
        "delta_vs_135": auc - roc_auc_score(y, CHAMPION_OOF)
    })

    print(
        f"{n_models:3d} models | "
        f"AUC={auc:.9f} | "
        f"Delta={auc - roc_auc_score(y, CHAMPION_OOF):+.9f}"
    )

print("\n" + "=" * 70)
print("FINE PRUNING RESULTS")
print("=" * 70)

print(
    pd.DataFrame(fine_results)
    .sort_values("auc", ascending=False)
    .to_string(index=False)
)

[2026-10-07 04:50:48.029] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:50:50.131] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:50:50.962] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:50:53.216] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
 65 models | AUC=0.962107625 | Delta=+0.000031466
[2026-10-07 04:50:55.505] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:50:56.639] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:50:57.743] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:50:58.350] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:50:59.200] [CUML] [warning] L-BFGS line search 

In [21]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from cuml.linear_model import LogisticRegression


# ============================================================
# PHASE 8F — FOLD-CONSISTENT PRUNING VALIDATION
# ============================================================

print("=" * 70)
print("PHASE 8F — FOLD-CONSISTENT PRUNING VALIDATION")
print("=" * 70)

PRUNE_SIZES = [135, 75, 50, 40]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {
    n: []
    for n in PRUNE_SIZES
}

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_oof, y)):

    print(f"\n{'='*25} FOLD {fold} {'='*25}")

    X_train = X_oof[train_idx]
    X_valid = X_oof[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    for n_models in PRUNE_SIZES:

        # ----------------------------------------------------
        # STEP 1: Fit full stack ONLY on training fold
        # ----------------------------------------------------

        base_model = LogisticRegression(
            penalty="l2",
            C=1.0,
            fit_intercept=True,
            solver="qn",
            max_iter=2000,
            tol=1e-7
        )

        base_model.fit(X_train, y_train)

        coef = np.asarray(
            base_model.coef_
        ).ravel()

        # ----------------------------------------------------
        # STEP 2: Select models using TRAINING fold only
        # ----------------------------------------------------

        ranked_idx = np.argsort(
            np.abs(coef)
        )[::-1]

        selected_idx = ranked_idx[:n_models]

        # ----------------------------------------------------
        # STEP 3: Refit selected stack
        # ----------------------------------------------------

        X_train_sel = X_train[:, selected_idx]
        X_valid_sel = X_valid[:, selected_idx]

        model = LogisticRegression(
            penalty="l2",
            C=1.0,
            fit_intercept=True,
            solver="qn",
            max_iter=2000,
            tol=1e-7
        )

        model.fit(X_train_sel, y_train)

        # ----------------------------------------------------
        # STEP 4: Predict VALIDATION fold
        # ----------------------------------------------------

        pred = model.predict_proba(X_valid_sel)[:, 1]

        auc = roc_auc_score(y_valid, pred)

        results[n_models].append(auc)

        print(
            f"{n_models:3d} models | "
            f"AUC = {auc:.9f}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PHASE 8F — FINAL RESULTS")
print("=" * 70)

summary = []

for n_models in PRUNE_SIZES:

    scores = np.array(results[n_models])

    summary.append({
        "models": n_models,
        "mean_auc": scores.mean(),
        "std": scores.std(),
        "min": scores.min(),
        "max": scores.max()
    })

summary_df = (
    pd.DataFrame(summary)
    .sort_values("mean_auc", ascending=False)
)

print(summary_df.to_string(index=False))

PHASE 8F — FOLD-CONSISTENT PRUNING VALIDATION

========================= FOLD 0 =========================
[2026-10-07 04:52:20.303] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:52:22.379] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
135 models | AUC = 0.962175727
[2026-10-07 04:52:24.278] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:52:25.419] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
 75 models | AUC = 0.962166722
[2026-10-07 04:52:27.123] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:52:28.104] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
 50 models | AUC = 0.962200386
[2026-10-07 04:52:29.585] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
[2026-10-07 04:52:30.099

In [22]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from cuml.linear_model import LogisticRegression

print("=" * 70)
print("PHASE 8G — STACK REGULARIZATION OPTIMIZATION")
print("=" * 70)

C_VALUES = [0.03, 0.1, 0.3, 1.0, 3.0]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for C in C_VALUES:

    fold_scores = []

    print(f"\n{'-'*60}")
    print(f"C = {C}")

    for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X_oof, y)
    ):

        X_train = X_oof[train_idx]
        X_valid = X_oof[valid_idx]

        y_train = y[train_idx]
        y_valid = y[valid_idx]

        model = LogisticRegression(
            penalty="l2",
            C=C,
            fit_intercept=True,
            solver="qn",
            max_iter=2000,
            tol=1e-7
        )

        model.fit(X_train, y_train)

        pred = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, pred)

        fold_scores.append(auc)

        print(
            f"Fold {fold}: {auc:.9f}"
        )

    fold_scores = np.array(fold_scores)

    mean_auc = fold_scores.mean()

    results.append({
        "C": C,
        "mean_auc": mean_auc,
        "std": fold_scores.std(),
        "min": fold_scores.min(),
        "max": fold_scores.max()
    })

    print(
        f"Mean AUC: {mean_auc:.9f}"
    )


results_df = (
    pd.DataFrame(results)
    .sort_values("mean_auc", ascending=False)
)

print("\n" + "=" * 70)
print("PHASE 8G — RESULTS")
print("=" * 70)

print(
    results_df.to_string(index=False)
)

PHASE 8G — STACK REGULARIZATION OPTIMIZATION

------------------------------------------------------------
C = 0.03
[2026-10-07 04:54:10.584] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 0: 0.962183182
[2026-10-07 04:54:12.034] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 1: 0.961193739
[2026-10-07 04:54:14.291] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 2: 0.962670434
[2026-10-07 04:54:16.098] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 3: 0.962335010
[2026-10-07 04:54:18.586] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 4: 0.962017281
Mean AUC: 0.962079929

------------------------------------------------------------
C = 0.1
[2026-10-07 04:54:20.410] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 0: 0.962183456
[2026-

In [23]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from cuml.linear_model import LogisticRegression

print("=" * 70)
print("PHASE 8H-1 — RANK TRANSFORM STACK")
print("=" * 70)

# ------------------------------------------------------------
# Rank-transform each model independently
# ------------------------------------------------------------

def rank_transform(X):

    X_rank = np.empty_like(X, dtype=np.float32)

    for j in range(X.shape[1]):

        order = np.argsort(X[:, j])

        ranks = np.empty(X.shape[0], dtype=np.float32)
        ranks[order] = (
            np.arange(X.shape[0], dtype=np.float32)
            / (X.shape[0] - 1)
        )

        X_rank[:, j] = ranks

    return X_rank


# X_oof currently contains logit-transformed predictions.
# Convert each column back to probability first.

X_prob = 1.0 / (1.0 + np.exp(-X_oof))

print("Creating rank representation...")

X_rank = rank_transform(X_prob)

print("Rank matrix:", X_rank.shape)


# ------------------------------------------------------------
# 5-fold validation
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(X_rank, y)
):

    X_train = X_rank[train_idx]
    X_valid = X_rank[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    model = LogisticRegression(
        penalty="l2",
        C=1.0,
        fit_intercept=True,
        solver="qn",
        max_iter=2000,
        tol=1e-7
    )

    model.fit(X_train, y_train)

    pred = model.predict_proba(X_valid)[:, 1]

    auc = roc_auc_score(y_valid, pred)

    fold_scores.append(auc)

    print(
        f"Fold {fold}: {auc:.9f}"
    )


fold_scores = np.array(fold_scores)

print("\n" + "=" * 70)
print("PHASE 8H-1 RESULT")
print("=" * 70)

print(
    f"Mean AUC : {fold_scores.mean():.9f}"
)

print(
    f"Std      : {fold_scores.std():.9f}"
)

print(
    f"Min      : {fold_scores.min():.9f}"
)

print(
    f"Max      : {fold_scores.max():.9f}"
)

PHASE 8H-1 — RANK TRANSFORM STACK
Creating rank representation...
Rank matrix: (699635, 135)
[2026-10-07 04:55:34.931] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 0: 0.961102611
[2026-10-07 04:55:38.471] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 1: 0.960305416
[2026-10-07 04:55:40.789] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 2: 0.961868198
[2026-10-07 04:55:44.328] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 3: 0.961167283
[2026-10-07 04:55:47.428] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 4: 0.960862989

PHASE 8H-1 RESULT
Mean AUC : 0.961061299
Std      : 0.000504918
Min      : 0.960305416
Max      : 0.961868198


In [24]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("PHASE 8I-1 — LIGHTGBM NONLINEAR META-STACK")
print("=" * 70)

# ============================================================
# IMPORTANT:
# X_oof = existing 135-model LOGIT prediction matrix
# y     = existing target
# ============================================================

print("X_oof shape:", X_oof.shape)
print("y shape    :", y.shape)

assert X_oof.shape[1] == 135


# ============================================================
# 5-FOLD VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(X_oof, y)
):

    print(f"\n{'='*25} FOLD {fold} {'='*25}")

    X_train = X_oof[train_idx]
    X_valid = X_oof[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    # --------------------------------------------------------
    # Small / regularized meta-model
    # --------------------------------------------------------

    model = lgb.LGBMClassifier(
        objective="binary",

        n_estimators=1000,
        learning_rate=0.02,

        num_leaves=15,
        max_depth=4,

        min_child_samples=100,

        subsample=0.8,
        colsample_bytree=0.8,

        reg_alpha=1.0,
        reg_lambda=5.0,

        random_state=42 + fold,

        n_jobs=-1,

        verbosity=-1
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (X_valid, y_valid)
        ],

        callbacks=[
            lgb.early_stopping(
                80,
                verbose=False
            )
        ]
    )

    pred = model.predict_proba(
        X_valid
    )[:, 1]

    auc = roc_auc_score(
        y_valid,
        pred
    )

    fold_scores.append(auc)

    print(
        f"Best iteration: {model.best_iteration_}"
    )

    print(
        f"Fold {fold} AUC: {auc:.9f}"
    )


# ============================================================
# FINAL RESULT
# ============================================================

fold_scores = np.array(fold_scores)

print("\n" + "=" * 70)
print("PHASE 8I-1 RESULT")
print("=" * 70)

print(
    f"Mean AUC : {fold_scores.mean():.9f}"
)

print(
    f"Std      : {fold_scores.std():.9f}"
)

print(
    f"Min      : {fold_scores.min():.9f}"
)

print(
    f"Max      : {fold_scores.max():.9f}"
)

print(
    f"Delta vs Logistic Stack : "
    f"{fold_scores.mean() - 0.962082:+.9f}"
)

PHASE 8I-1 — LIGHTGBM NONLINEAR META-STACK
X_oof shape: (699635, 135)
y shape    : (699635,)

========================= FOLD 0 =========================


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Best iteration: 417
Fold 0 AUC: 0.962063703

========================= FOLD 1 =========================


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Best iteration: 348
Fold 1 AUC: 0.961096588

========================= FOLD 2 =========================


...

Best iteration: 390
Fold 2 AUC: 0.962560985

========================= FOLD 3 =========================


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Best iteration: 388
Fold 3 AUC: 0.962192453

========================= FOLD 4 =========================


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Best iteration: 546
Fold 4 AUC: 0.961932136

PHASE 8I-1 RESULT
Mean AUC : 0.961969173
Std      : 0.000484109
Min      : 0.961096588
Max      : 0.962560985
Delta vs Logistic Stack : -0.000112827


In [26]:
print("Available variables:")
for v in ["X_oof", "X_test", "stack_oof", "stack_test", "model_names"]:
    print(f"{v:15} ->", "YES" if v in globals() else "NO")

print("\nX_oof shape:", X_oof.shape if "X_oof" in globals() else "N/A")

Available variables:
X_oof           -> YES
X_test          -> YES
stack_oof       -> YES
stack_test      -> YES
model_names     -> NO

X_oof shape: (699635, 135)


In [27]:
print("Variables containing 'name', 'model', or 'oof':")

for v in sorted(globals()):
    vl = v.lower()
    if "name" in vl or "model" in vl or "oof" in vl:
        obj = globals()[v]
        try:
            print(f"{v:30} {type(obj).__name__}  shape={obj.shape}")
        except:
            print(f"{v:30} {type(obj).__name__}")

Variables containing 'name', 'model', or 'oof':
CHAMPION_OOF                   ndarray  shape=(699635,)
X_oof                          ndarray  shape=(699635, 135)
__name__                       str
base_model                     LogisticRegression
model                          LGBMClassifier
models                         DataFrame  shape=(135, 4)
n_models                       int
name                           str
names                          list
nj_oof                         ndarray  shape=(699635,)
oof                            ndarray  shape=(699635,)
oof_pred                       ndarray  shape=(699635,)
oof_stack                      ndarray  shape=(699635,)
stack_oof                      ndarray  shape=(699635,)


In [29]:
print("=" * 70)
print("MODEL NAME MAPPING CHECK")
print("=" * 70)

print("\nmodels columns:")
print(models.columns.tolist())

print("\nmodels shape:")
print(models.shape)

print("\nFirst 10 rows:")
print(models.head(10).to_string(index=False))

print("\nnames length:", len(names))
print("First 20 names:")
print(names[:20])

print("\nLast 10 names:")
print(names[-10:])

MODEL NAME MAPPING CHECK

models columns:
['model', 'dataset', 'already_logit', 'oof_auc']

models shape:
(135, 4)

First 10 rows:
        model                                   dataset  already_logit  oof_auc
    ar_cat_5f s6e10-124-public-oof-and-test-predictions           True 0.960940
    ar_lgb_5f s6e10-124-public-oof-and-test-predictions           True 0.961029
ar_realmlp_5f s6e10-124-public-oof-and-test-predictions           True 0.960879
    ar_xgb_5f s6e10-124-public-oof-and-test-predictions           True 0.961049
       bp_cat s6e10-124-public-oof-and-test-predictions          False 0.960887
      bp_lgbm s6e10-124-public-oof-and-test-predictions          False 0.961058
   bp_lgbm_xt s6e10-124-public-oof-and-test-predictions          False 0.960930
   bp_realmlp s6e10-124-public-oof-and-test-predictions          False 0.961216
       bp_xgb s6e10-124-public-oof-and-test-predictions          False 0.961020
         gl_a s6e10-124-public-oof-and-test-predictions          Fals

In [30]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("PHASE 9A — RESIDUAL / HARD-CASE COMPLEMENTARITY ANALYSIS")
print("=" * 70)

# ============================================================
# 1. Confirm mapping
# ============================================================
model_names = names

assert X_oof.shape[1] == len(model_names), \
    f"Mismatch: X_oof has {X_oof.shape[1]} columns, names has {len(model_names)}"

print(f"\nModels : {len(model_names)}")
print(f"Rows   : {X_oof.shape[0]}")

# ============================================================
# 2. Current champion
# ============================================================
champ = np.asarray(stack_oof).ravel()
y_arr = np.asarray(y).astype(int)

champ_auc = roc_auc_score(y_arr, champ)

print(f"\nChampion OOF AUC : {champ_auc:.9f}")

# ============================================================
# 3. Convert X_oof back to probabilities
#
# X_oof contains logits for the 135 models.
# ============================================================
def sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))

base_prob = sigmoid(X_oof)

print(f"Base prediction matrix : {base_prob.shape}")

# ============================================================
# 4. Champion residual
# ============================================================
champ_resid = y_arr - champ

champ_abs_error = np.abs(champ_resid)

print("\nChampion error:")
print(f"Mean absolute error : {champ_abs_error.mean():.6f}")
print(f"Std                 : {champ_abs_error.std():.6f}")

# ============================================================
# 5. Hard cases = top 10% largest champion errors
# ============================================================
threshold = np.quantile(champ_abs_error, 0.90)

hard_mask = champ_abs_error >= threshold

print("\nHard-case definition:")
print(f"Threshold : {threshold:.6f}")
print(f"Cases     : {hard_mask.sum():,}")
print(f"Fraction  : {hard_mask.mean():.2%}")

# ============================================================
# 6. Analyze each base model
# ============================================================
results = []

for i, name in enumerate(model_names):

    p = base_prob[:, i]

    # Full OOF AUC
    auc = roc_auc_score(y_arr, p)

    # AUC specifically on champion-hard cases
    hard_auc = roc_auc_score(
        y_arr[hard_mask],
        p[hard_mask]
    )

    # Model residual
    resid = y_arr - p

    # Residual correlation with champion
    resid_corr = np.corrcoef(
        champ_resid,
        resid
    )[0, 1]

    # Prediction correlation with champion
    pred_corr = np.corrcoef(
        champ,
        p
    )[0, 1]

    # Difference from champion
    mean_abs_diff = np.mean(np.abs(p - champ))
    residual_diff_std = np.std(resid - champ_resid)

    results.append({
        "model": name,
        "auc": auc,
        "hard_auc": hard_auc,
        "hard_delta": hard_auc - auc,
        "resid_corr": resid_corr,
        "pred_corr": pred_corr,
        "mean_abs_diff": mean_abs_diff,
        "residual_diff_std": residual_diff_std
    })

res_df = pd.DataFrame(results)

# ============================================================
# 7. TOP HARD-CASE MODELS
# ============================================================
print("\n" + "=" * 70)
print("TOP 20 — HARD-CASE AUC")
print("=" * 70)

print(
    res_df
    .sort_values("hard_auc", ascending=False)
    .head(20)
    .to_string(index=False)
)

# ============================================================
# 8. MODELS THAT IMPROVE MOST ON HARD CASES
# ============================================================
print("\n" + "=" * 70)
print("TOP 20 — HARD-CASE DELTA")
print("=" * 70)

print(
    res_df
    .sort_values("hard_delta", ascending=False)
    .head(20)
    .to_string(index=False)
)

# ============================================================
# 9. LOWEST RESIDUAL CORRELATION
# ============================================================
print("\n" + "=" * 70)
print("TOP 20 — LOWEST RESIDUAL CORRELATION")
print("=" * 70)

print(
    res_df
    .sort_values("resid_corr", ascending=True)
    .head(20)
    .to_string(index=False)
)

# ============================================================
# 10. Composite complementarity score
#
# Higher hard-case AUC is good.
# Lower residual correlation is good.
#
# This is ONLY for candidate discovery.
# We will NOT treat this as final validation.
# ============================================================
res_df["complement_score"] = (
    res_df["hard_auc"]
    - 0.25 * res_df["resid_corr"]
)

print("\n" + "=" * 70)
print("TOP 20 — COMPLEMENTARITY SCORE")
print("=" * 70)

print(
    res_df
    .sort_values("complement_score", ascending=False)
    .head(20)
    .to_string(index=False)
)

PHASE 9A — RESIDUAL / HARD-CASE COMPLEMENTARITY ANALYSIS

Models : 135
Rows   : 699635

Champion OOF AUC : 0.962076160
Base prediction matrix : (699635, 135)

Champion error:
Mean absolute error : 0.117588
Std                 : 0.212087

Hard-case definition:
Threshold : 0.296627
Cases     : 69,964
Fraction  : 10.00%

TOP 20 — HARD-CASE AUC
             model      auc  hard_auc  hard_delta  resid_corr  pred_corr  mean_abs_diff  residual_diff_std
              gl_f 0.948865  0.180954   -0.767911    0.888967   0.957721       0.069537           0.124907
              gl_g 0.954592  0.159183   -0.795409    0.933961   0.978615       0.052642           0.092778
              gl_e 0.956817  0.152647   -0.804171    0.973855   0.991459       0.027702           0.056563
            mitra2 0.957075  0.150211   -0.806864    0.982851   0.994484       0.023010           0.045485
gp_tabicl_full_raw 0.957590  0.149470   -0.808120    0.981278   0.994397       0.024804           0.047586
              g

In [31]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

print("=" * 70)
print("PHASE 9B — FOLD-SAFE RESIDUAL CANDIDATE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Candidate models
# ------------------------------------------------------------
candidate_names = [
    "ar_realmlp_5f",
    "ar_xgb_5f",
    "ar_lgb_5f",
    "ar_cat_5f",
    "nj_01"
]

name_to_idx = {
    name: i for i, name in enumerate(model_names)
}

candidate_idx = [
    name_to_idx[name]
    for name in candidate_names
]

print("\nCandidates:")
for name, idx in zip(candidate_names, candidate_idx):
    print(f"{name:20} -> column {idx}")

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
X = X_oof.astype(np.float32)
y_arr = np.asarray(y).astype(np.int32)

# Candidate predictions are currently logits
candidate_prob = sigmoid(
    X[:, candidate_idx]
)

# ------------------------------------------------------------
# IMPORTANT:
# Use the existing champion OOF as the baseline.
#
# The champion itself is already OOF.
# We use folds only to evaluate candidate blends
# without selecting weights from the evaluation fold.
# ------------------------------------------------------------
champ = np.asarray(CHAMPION_OOF).ravel()

# ------------------------------------------------------------
# Fixed weights
# ------------------------------------------------------------
weights = [
    0.000,
    0.0025,
    0.0050,
    0.0100,
    0.0200,
    0.0300,
    0.0500
]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for candidate_pos, candidate_name in enumerate(candidate_names):

    candidate = candidate_prob[:, candidate_pos]

    print("\n" + "-" * 70)
    print(f"Candidate: {candidate_name}")
    print("-" * 70)

    fold_scores = {
        w: []
        for w in weights
    }

    for fold, (_, valid_idx) in enumerate(
        skf.split(X, y_arr)
    ):

        y_valid = y_arr[valid_idx]

        champ_valid = champ[valid_idx]
        cand_valid = candidate[valid_idx]

        for w in weights:

            if w == 0:
                blend = champ_valid
            else:
                # Convex probability blend
                blend = (
                    (1.0 - w) * champ_valid
                    + w * cand_valid
                )

            auc = roc_auc_score(
                y_valid,
                blend
            )

            fold_scores[w].append(auc)

    for w in weights:

        scores = np.array(fold_scores[w])

        results.append({
            "candidate": candidate_name,
            "weight": w,
            "mean_auc": scores.mean(),
            "std": scores.std(),
            "min_auc": scores.min(),
            "max_auc": scores.max()
        })

        print(
            f"weight={w:7.4f} | "
            f"mean={scores.mean():.9f} | "
            f"std={scores.std():.6f}"
        )

# ------------------------------------------------------------
# Results dataframe
# ------------------------------------------------------------
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("BEST WEIGHT PER CANDIDATE")
print("=" * 70)

best_rows = (
    results_df
    .sort_values(
        ["candidate", "mean_auc"],
        ascending=[True, False]
    )
    .groupby("candidate")
    .head(1)
)

print(
    best_rows
    .sort_values("mean_auc", ascending=False)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Overall baseline
# ------------------------------------------------------------
baseline_auc = results_df[
    results_df["weight"] == 0
]["mean_auc"].mean()

print("\n" + "=" * 70)
print("BASELINE")
print("=" * 70)

print(f"Champion fold CV mean AUC : {baseline_auc:.9f}")

print("\nDelta vs champion:")

best_rows = best_rows.copy()
best_rows["delta"] = (
    best_rows["mean_auc"] - baseline_auc
)

print(
    best_rows[
        ["candidate", "weight", "mean_auc", "delta"]
    ]
    .sort_values("delta", ascending=False)
    .to_string(index=False)
)

PHASE 9B — FOLD-SAFE RESIDUAL CANDIDATE VALIDATION

Candidates:
ar_realmlp_5f        -> column 2
ar_xgb_5f            -> column 3
ar_lgb_5f            -> column 1
ar_cat_5f            -> column 0
nj_01                -> column 93

----------------------------------------------------------------------
Candidate: ar_realmlp_5f
----------------------------------------------------------------------
weight= 0.0000 | mean=0.962081401 | std=0.000482
weight= 0.0025 | mean=0.962081949 | std=0.000482
weight= 0.0050 | mean=0.962082491 | std=0.000483
weight= 0.0100 | mean=0.962083465 | std=0.000483
weight= 0.0200 | mean=0.962084926 | std=0.000485
weight= 0.0300 | mean=0.962085716 | std=0.000486
weight= 0.0500 | mean=0.962085731 | std=0.000488

----------------------------------------------------------------------
Candidate: ar_xgb_5f
----------------------------------------------------------------------
weight= 0.0000 | mean=0.962081401 | std=0.000482
weight= 0.0025 | mean=0.962082438 | std=0.0004

In [32]:
import numpy as np
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("PHASE 9C — NJ_01 FINE WEIGHT SEARCH")
print("=" * 70)

champ = np.asarray(CHAMPION_OOF).ravel()

nj_idx = model_names.index("nj_01")
nj_pred = sigmoid(X_oof[:, nj_idx])

# Fine grid around the observed optimum
weights = np.arange(
    0.010,
    0.0501,
    0.0025
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for w in weights:

    fold_scores = []

    for _, valid_idx in skf.split(X_oof, y_arr):

        blend = (
            (1.0 - w) * champ[valid_idx]
            + w * nj_pred[valid_idx]
        )

        auc = roc_auc_score(
            y_arr[valid_idx],
            blend
        )

        fold_scores.append(auc)

    fold_scores = np.array(fold_scores)

    results.append({
        "weight": w,
        "mean_auc": fold_scores.mean(),
        "std": fold_scores.std(),
        "min_auc": fold_scores.min(),
        "max_auc": fold_scores.max()
    })

fine_df = pd.DataFrame(results)

baseline = np.mean([
    roc_auc_score(
        y_arr[idx],
        champ[idx]
    )
    for _, idx in skf.split(X_oof, y_arr)
])

fine_df["delta"] = fine_df["mean_auc"] - baseline

print(
    fine_df
    .sort_values("mean_auc", ascending=False)
    .to_string(index=False)
)

print("\n" + "=" * 70)
print("BEST NJ_01 WEIGHT")
print("=" * 70)

best = fine_df.loc[
    fine_df["mean_auc"].idxmax()
]

print(best.to_string())

print("\nBaseline:")
print(f"{baseline:.9f}")

print("\nBest improvement:")
print(f"{best['delta']:+.9f}")

PHASE 9C — NJ_01 FINE WEIGHT SEARCH
 weight  mean_auc      std  min_auc  max_auc    delta
 0.0275  0.962096 0.000494 0.961207 0.962697 0.000015
 0.0300  0.962096 0.000495 0.961205 0.962698 0.000014
 0.0250  0.962096 0.000493 0.961209 0.962695 0.000014
 0.0325  0.962096 0.000495 0.961203 0.962700 0.000014
 0.0225  0.962096 0.000492 0.961210 0.962694 0.000014
 0.0350  0.962096 0.000496 0.961201 0.962701 0.000014
 0.0375  0.962095 0.000497 0.961200 0.962702 0.000014
 0.0200  0.962095 0.000491 0.961212 0.962691 0.000014
 0.0400  0.962095 0.000498 0.961198 0.962703 0.000013
 0.0175  0.962094 0.000490 0.961213 0.962689 0.000013
 0.0425  0.962094 0.000499 0.961196 0.962704 0.000013
 0.0450  0.962094 0.000499 0.961194 0.962704 0.000012
 0.0150  0.962094 0.000489 0.961214 0.962686 0.000012
 0.0475  0.962093 0.000500 0.961191 0.962705 0.000012
 0.0125  0.962092 0.000488 0.961215 0.962683 0.000011
 0.0500  0.962092 0.000501 0.961189 0.962705 0.000011
 0.0100  0.962091 0.000487 0.961216 0.962680 0

In [33]:
print("=" * 70)
print("PHASE 9D — NJ_01 + AR_XGB COMPLEMENTARY BLEND")
print("=" * 70)

champ = np.asarray(CHAMPION_OOF).ravel()

# ------------------------------------------------------------
# Candidate indices
# ------------------------------------------------------------
nj_idx = model_names.index("nj_01")
arx_idx = model_names.index("ar_xgb_5f")

nj_pred = sigmoid(X_oof[:, nj_idx])
arx_pred = sigmoid(X_oof[:, arx_idx])

print("\nCandidates:")
print(f"NJ_01       -> column {nj_idx}")
print(f"ar_xgb_5f   -> column {arx_idx}")

# ------------------------------------------------------------
# Fixed grids
# ------------------------------------------------------------
nj_weights = [
    0.015,
    0.020,
    0.025,
    0.0275,
    0.030,
    0.035,
    0.040
]

arx_weights = [
    0.010,
    0.020,
    0.030,
    0.040,
    0.050
]

# ------------------------------------------------------------
# Same fixed 5 folds
# ------------------------------------------------------------
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Precompute folds
folds = list(skf.split(X_oof, y_arr))

results = []

# ------------------------------------------------------------
# Test combinations
# ------------------------------------------------------------
for nj_w in nj_weights:

    for arx_w in arx_weights:

        # Keep total candidate weight <= 9%
        if nj_w + arx_w > 0.09:
            continue

        fold_scores = []

        for _, valid_idx in folds:

            y_valid = y_arr[valid_idx]

            c = champ[valid_idx]
            nj = nj_pred[valid_idx]
            arx = arx_pred[valid_idx]

            blend = (
                (1.0 - nj_w - arx_w) * c
                + nj_w * nj
                + arx_w * arx
            )

            auc = roc_auc_score(
                y_valid,
                blend
            )

            fold_scores.append(auc)

        fold_scores = np.asarray(fold_scores)

        results.append({
            "nj_weight": nj_w,
            "arx_weight": arx_w,
            "mean_auc": fold_scores.mean(),
            "std": fold_scores.std(),
            "min_auc": fold_scores.min(),
            "max_auc": fold_scores.max()
        })

results_9d = pd.DataFrame(results)

# ------------------------------------------------------------
# Baseline
# ------------------------------------------------------------
champ_fold_scores = []

for _, valid_idx in folds:

    champ_fold_scores.append(
        roc_auc_score(
            y_arr[valid_idx],
            champ[valid_idx]
        )
    )

baseline = np.mean(champ_fold_scores)

results_9d["delta"] = (
    results_9d["mean_auc"] - baseline
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("TOP 15 COMBINATIONS")
print("=" * 70)

print(
    results_9d
    .sort_values("mean_auc", ascending=False)
    .head(15)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Best
# ------------------------------------------------------------
best = results_9d.loc[
    results_9d["mean_auc"].idxmax()
]

print("\n" + "=" * 70)
print("BEST COMBINATION")
print("=" * 70)

print(best.to_string())

print("\nChampion:")
print(f"{baseline:.9f}")

print("\nImprovement:")
print(f"{best['delta']:+.9f}")

PHASE 9D — NJ_01 + AR_XGB COMPLEMENTARY BLEND

Candidates:
NJ_01       -> column 93
ar_xgb_5f   -> column 3

TOP 15 COMBINATIONS
 nj_weight  arx_weight  mean_auc      std  min_auc  max_auc    delta
    0.0150        0.05  0.962099 0.000498 0.961210 0.962719 0.000018
    0.0200        0.04  0.962099 0.000498 0.961208 0.962716 0.000017
    0.0200        0.05  0.962099 0.000500 0.961206 0.962721 0.000017
    0.0150        0.04  0.962099 0.000496 0.961211 0.962714 0.000017
    0.0200        0.03  0.962099 0.000496 0.961209 0.962711 0.000017
    0.0250        0.04  0.962099 0.000500 0.961204 0.962718 0.000017
    0.0250        0.03  0.962098 0.000498 0.961206 0.962713 0.000017
    0.0250        0.05  0.962098 0.000501 0.961202 0.962723 0.000017
    0.0150        0.03  0.962098 0.000494 0.961213 0.962708 0.000017
    0.0275        0.04  0.962098 0.000500 0.961202 0.962719 0.000017
    0.0275        0.03  0.962098 0.000499 0.961204 0.962714 0.000017
    0.0250        0.02  0.962098 0.000496 0

In [34]:
print("=" * 70)
print("PHASE 9E — LOGIT-SPACE COMPLEMENTARY BLEND")
print("=" * 70)

from scipy.special import logit, expit

champ = np.asarray(CHAMPION_OOF).ravel()

nj_idx = model_names.index("nj_01")
arx_idx = model_names.index("ar_xgb_5f")

nj_prob = sigmoid(X_oof[:, nj_idx])
arx_prob = sigmoid(X_oof[:, arx_idx])

# Safe logits
champ_logit = logit(np.clip(champ, 1e-7, 1 - 1e-7))
nj_logit = logit(np.clip(nj_prob, 1e-7, 1 - 1e-7))
arx_logit = logit(np.clip(arx_prob, 1e-7, 1 - 1e-7))

# Candidate grids
nj_weights = [
    0.005,
    0.010,
    0.015,
    0.020,
    0.025,
    0.030,
]

arx_weights = [
    0.010,
    0.020,
    0.030,
    0.040,
    0.050,
    0.060,
]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(skf.split(X_oof, y_arr))

# Champion baseline
champ_scores = []

for _, valid_idx in folds:
    champ_scores.append(
        roc_auc_score(
            y_arr[valid_idx],
            champ[valid_idx]
        )
    )

baseline = np.mean(champ_scores)

results = []

for nj_w in nj_weights:

    for arx_w in arx_weights:

        if nj_w + arx_w > 0.10:
            continue

        fold_scores = []

        for _, valid_idx in folds:

            z = (
                (1.0 - nj_w - arx_w) * champ_logit[valid_idx]
                + nj_w * nj_logit[valid_idx]
                + arx_w * arx_logit[valid_idx]
            )

            blend = expit(z)

            auc = roc_auc_score(
                y_arr[valid_idx],
                blend
            )

            fold_scores.append(auc)

        fold_scores = np.asarray(fold_scores)

        results.append({
            "nj_weight": nj_w,
            "arx_weight": arx_w,
            "mean_auc": fold_scores.mean(),
            "std": fold_scores.std(),
            "min_auc": fold_scores.min(),
            "max_auc": fold_scores.max(),
            "delta": fold_scores.mean() - baseline
        })

logit_results = pd.DataFrame(results)

print("\n" + "=" * 70)
print("TOP 15 LOGIT-SPACE COMBINATIONS")
print("=" * 70)

print(
    logit_results
    .sort_values("mean_auc", ascending=False)
    .head(15)
    .to_string(index=False)
)

best = logit_results.loc[
    logit_results["mean_auc"].idxmax()
]

print("\n" + "=" * 70)
print("BEST LOGIT-SPACE COMBINATION")
print("=" * 70)

print(best.to_string())

print("\nChampion:")
print(f"{baseline:.9f}")

print("\nImprovement:")
print(f"{best['delta']:+.9f}")

PHASE 9E — LOGIT-SPACE COMPLEMENTARY BLEND

TOP 15 LOGIT-SPACE COMBINATIONS
 nj_weight  arx_weight  mean_auc      std  min_auc  max_auc    delta
     0.030        0.06  0.962093 0.000488 0.961217 0.962688 0.000011
     0.030        0.05  0.962092 0.000488 0.961217 0.962686 0.000011
     0.025        0.06  0.962092 0.000488 0.961217 0.962686 0.000010
     0.030        0.04  0.962092 0.000487 0.961217 0.962685 0.000010
     0.025        0.05  0.962091 0.000487 0.961217 0.962684 0.000010
     0.020        0.06  0.962091 0.000487 0.961218 0.962684 0.000010
     0.030        0.03  0.962091 0.000487 0.961217 0.962683 0.000009
     0.025        0.04  0.962091 0.000487 0.961217 0.962682 0.000009
     0.020        0.05  0.962090 0.000487 0.961217 0.962682 0.000009
     0.030        0.02  0.962090 0.000487 0.961217 0.962681 0.000009
     0.025        0.03  0.962090 0.000486 0.961217 0.962680 0.000009
     0.015        0.06  0.962090 0.000486 0.961218 0.962681 0.000008
     0.020        0.04  0.9